# UD2.03. Procesamiento de texto con Azure AI Language

**Módulo 5073 · Programación de Inteligencia Artificial · UD2 · Práctica P2.2**

Azure AI Language expone **todas** sus tareas de texto en el mismo endpoint. Cambia un campo del
cuerpo, `kind`, y tienes seis servicios distintos. Aprendes una llamada y los tienes todos.

```
POST {endpoint}/language/:analyze-text?api-version=2022-05-01
```

| `kind` | Qué hace | Modo |
|---|---|---|
| `LanguageDetection` | Idioma y confianza | Síncrono |
| `SentimentAnalysis` | Positivo, neutro o negativo, con puntuaciones | Síncrono |
| `KeyPhraseExtraction` | Frases clave | Síncrono |
| `EntityRecognition` | Personas, lugares, organizaciones, fechas | Síncrono |
| `PiiEntityRecognition` | Datos personales, con el texto enmascarado | Síncrono |
| `ExtractiveSummarization` | Resumen seleccionando frases | **Asíncrono** |

Este cuaderno construye el módulo `servicios/lenguaje.py` de la práctica P2.2, función a función.
Al final lo tendrás entero y probado.

> **Antes de ejecutar nada**, ten hecho el cuaderno **UD2.02**: la clave va en el `.env`, no aquí.

In [ ]:
!pip install -q requests python-dotenv

In [ ]:
import os
import time
from typing import Sequence

import requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

CLAVE = os.getenv("AZURE_LANGUAGE_KEY")
ENDPOINT = (os.getenv("AZURE_LANGUAGE_ENDPOINT") or "").rstrip("/")

print("Clave cargada:  ", bool(CLAVE))
print("Endpoint:       ", ENDPOINT or "(sin definir)")

if not (CLAVE and ENDPOINT):
    print("\nFalta configuración. Repasa el cuaderno UD2.02 antes de seguir.")

## 1. La estructura del cuerpo, y el error más caro de la unidad

Todas las tareas síncronas comparten esta forma:

```python
{
  "kind": "SentimentAnalysis",
  "parameters": {"modelVersion": "latest"},
  "analysisInput": {
      "documents": [
          {"id": "1", "language": "es", "text": "..."},
          {"id": "2", "language": "es", "text": "..."}
      ]
  }
}
```

Dos cosas de esa estructura no son detalles:

**Se envían varios documentos en una sola petición.** Lo que se factura es la llamada, y el
límite de peticiones por segundo se cuenta por llamada. Analizar 500 reseñas de una en una son
500 llamadas, un 429 seguro y una factura innecesaria.

**Cada documento lleva un `id` que eliges tú**, y la respuesta te lo devuelve. Hace falta porque
**el orden de los resultados no está garantizado**: se casa por `id`, nunca por posición. Es un
error que no se nota mientras pruebas con tres documentos y aparece en producción con doscientos.

## 2. La llamada base

Una sola función hace la petición. Las seis tareas son la misma llamada con otro `kind`.

Reutiliza las excepciones del cuaderno UD2.01: quien llame decide qué enseñar al usuario, este
módulo no imprime nada.

In [ ]:
class ErrorServicio(Exception):
    """Cualquier fallo al hablar con un servicio remoto."""


class ErrorAutenticacion(ErrorServicio):
    """401 o 403."""


class ErrorCuota(ErrorServicio):
    """429: demasiadas peticiones."""


class ErrorPeticion(ErrorServicio):
    """Resto de 4xx: la petición está mal construida."""


class ErrorProveedor(ErrorServicio):
    """5xx o fallo de red."""


URL_TEXTO = f"{ENDPOINT}/language/:analyze-text?api-version=2022-05-01"
CABECERAS = {"Ocp-Apim-Subscription-Key": CLAVE or "", "Content-Type": "application/json"}


def _comprueba(respuesta):
    """Traduce el código de estado a una excepción con mensaje legible."""
    codigo = respuesta.status_code
    if codigo == 200:
        return respuesta.json()
    if codigo in (401, 403):
        raise ErrorAutenticacion(f"Credenciales rechazadas (HTTP {codigo})")
    if codigo == 429:
        raise ErrorCuota(
            "Límite de peticiones superado. "
            f"Retry-After: {respuesta.headers.get('Retry-After', 'no indicado')}"
        )
    if 400 <= codigo < 500:
        raise ErrorPeticion(f"Petición incorrecta (HTTP {codigo}): {respuesta.text[:300]}")
    raise ErrorProveedor(f"Fallo del proveedor (HTTP {codigo})")


def _analiza(kind, textos, idioma="es", parametros=None, timeout=20):
    """Llamada base a analyze-text. Devuelve la respuesta JSON completa.

    textos: lista de cadenas. Se numeran por posición para poder casarlas después.
    """
    documentos = []
    for i, texto in enumerate(textos, start=1):
        documento = {"id": str(i), "text": texto}
        if idioma:                      # LanguageDetection no lleva language
            documento["language"] = idioma
        documentos.append(documento)
    cuerpo = {
        "kind": kind,
        "parameters": parametros or {"modelVersion": "latest"},
        "analysisInput": {"documents": documentos},
    }
    try:
        respuesta = requests.post(URL_TEXTO, headers=CABECERAS, json=cuerpo, timeout=timeout)
    except requests.Timeout as error:
        raise ErrorProveedor(f"El servicio no respondió en {timeout} s") from error
    except requests.RequestException as error:
        raise ErrorProveedor(f"No se pudo contactar: {error}") from error

    return _comprueba(respuesta)

### Casar por `id` y no perder los errores

La respuesta trae `documents` con los éxitos y `errors` con los fallos, **mezclados en la misma
llamada**. Si envías cuatro documentos y uno es demasiado largo, recibes tres resultados y un
error, y un 200 en la cabecera.

Devolver solo los tres callado es un fallo que se corrige. Esta función pone cada resultado en su
sitio de la lista original y deja el error en su hueco.

In [ ]:
def _ordena(respuesta, total):
    """Devuelve una lista de longitud `total`, con el resultado o el error de cada documento."""
    resultados = [None] * total
    bloque = respuesta["results"]

    for documento in bloque.get("documents", []):
        resultados[int(documento["id"]) - 1] = documento

    for fallo in bloque.get("errors", []):
        resultados[int(fallo["id"]) - 1] = {
            "error": fallo["error"].get("message", "error sin mensaje"),
            "codigo": fallo["error"].get("code"),
        }

    return resultados

## 3. Detección de idioma

La tarea más barata, y la que conviene ejecutar primero cuando no sabes en qué idioma viene el
texto: el resto de tareas dan peor resultado si les declaras un idioma equivocado.

In [ ]:
def detecta_idioma(textos: Sequence[str]) -> list[dict]:
    """Idioma detectado de cada texto, con su código ISO y la confianza."""
    respuesta = _analiza("LanguageDetection", textos, idioma=None)
    salida = []
    for item in _ordena(respuesta, len(textos)):
        if item is None or "error" in (item or {}):
            salida.append(item or {"error": "sin respuesta"})
            continue
        detectado = item["detectedLanguage"]
        salida.append({
            "idioma": detectado["name"],
            "iso": detectado["iso6391Name"],
            "confianza": detectado["confidenceScore"],
        })
    return salida

Ojo a un detalle: en `LanguageDetection` no se manda `language` en el documento, porque el idioma
es justo lo que se pregunta. Por eso la función pasa `idioma=None` y `_analiza` **omite la clave**
en lugar de mandarla vacía. Enviar `"language": null` no es lo mismo que no enviarla, y el
servicio responde 400.

In [ ]:
TEXTOS = [
    "El servicio de atención al ciudadano ha sido excelente y muy rápido.",
    "Bon dia, voldria demanar cita per a renovar el DNI.",
    "This document is written in English.",
]

if CLAVE and ENDPOINT:
    for texto, resultado in zip(TEXTOS, detecta_idioma(TEXTOS)):
        print(f"{texto[:45]:47} -> {resultado}")

**Mira con atención el resultado del texto en valenciano.** Azure lo devolverá como catalán
(`ca`), que es lo correcto desde el punto de vista del estándar ISO 639-1, donde el valenciano no
tiene código propio.

Eso es exactamente el tipo de hallazgo que pide la actividad A2.1 cuando dice "calidad en tu
idioma": no basta con que el servicio soporte el idioma, hay que saber **con qué etiqueta** lo
devuelve y si eso encaja con lo que tu aplicación espera.

## 4. Análisis de sentimiento

Devuelve una etiqueta global y las tres puntuaciones. Además, con `opinionMining`, puede decir
qué opinión concreta se tiene sobre qué aspecto: "la comida buena, el servicio lento".

In [ ]:
def analiza_sentimiento(textos: Sequence[str], idioma: str = "es",
                        opiniones: bool = False) -> list[dict]:
    """Sentimiento global de cada texto y las tres puntuaciones."""
    parametros = {"modelVersion": "latest"}
    if opiniones:
        parametros["opinionMining"] = True

    respuesta = _analiza("SentimentAnalysis", textos, idioma, parametros)
    salida = []
    for item in _ordena(respuesta, len(textos)):
        if item is None or "error" in (item or {}):
            salida.append(item or {"error": "sin respuesta"})
            continue
        salida.append({
            "sentimiento": item["sentiment"],
            "positivo": item["confidenceScores"]["positive"],
            "neutro": item["confidenceScores"]["neutral"],
            "negativo": item["confidenceScores"]["negative"],
            "frases": [
                {"texto": f["text"], "sentimiento": f["sentiment"]}
                for f in item.get("sentences", [])
            ],
        })
    return salida

In [ ]:
RESENAS = [
    "El trámite fue rapidísimo y el personal muy amable. Repetiría sin dudarlo.",
    "Llevo tres semanas esperando respuesta y nadie me coge el teléfono.",
    "La cita era a las diez y me atendieron a las diez y cuarto.",
]

if CLAVE and ENDPOINT:
    for resena, r in zip(RESENAS, analiza_sentimiento(RESENAS)):
        print(f"{r['sentimiento']:9} (pos {r['positivo']:.2f} / neg {r['negativo']:.2f})  "
              f"{resena[:50]}")

Fíjate en la tercera. Un texto que a una persona le suena a queja leve suele salir **neutro** con
puntuaciones repartidas, porque no contiene ninguna palabra de valoración.

Es la razón por la que la confianza debe mostrarse siempre en la interfaz: un "neutro" al 45 % y
un "negativo" al 97 % no significan lo mismo, y ocultar el número induce a error a quien usa la
aplicación. Está en la rúbrica de PR2.

## 5. Frases clave y entidades

Dos tareas que se parecen y no son lo mismo:

- **Frases clave**: los conceptos importantes del texto, tal como aparecen. No están clasificados.
- **Entidades**: fragmentos identificados **y categorizados**: persona, lugar, organización,
  fecha, cantidad.

Para un buscador quieres frases clave. Para rellenar una ficha estructurada, entidades.

In [ ]:
def extrae_frases_clave(textos: Sequence[str], idioma: str = "es") -> list[list[str]]:
    """Frases clave de cada texto."""
    respuesta = _analiza("KeyPhraseExtraction", textos, idioma)
    return [
        item.get("keyPhrases", []) if item and "error" not in item else []
        for item in _ordena(respuesta, len(textos))
    ]


def reconoce_entidades(textos: Sequence[str], idioma: str = "es") -> list[list[dict]]:
    """Entidades de cada texto, con categoría y confianza."""
    respuesta = _analiza("EntityRecognition", textos, idioma)
    salida = []
    for item in _ordena(respuesta, len(textos)):
        if not item or "error" in item:
            salida.append([])
            continue
        salida.append([
            {
                "texto": e["text"],
                "categoria": e["category"],
                "subcategoria": e.get("subcategory"),
                "confianza": e["confidenceScore"],
            }
            for e in item.get("entities", [])
        ])
    return salida

In [ ]:
ACTA = ("El pleno del Ayuntamiento de Gandia aprobó el 12 de marzo de 2026 una partida "
        "de 240.000 euros para la digitalización del archivo municipal, a propuesta de "
        "la concejalía de Modernización.")

if CLAVE and ENDPOINT:
    print("Frases clave:", extrae_frases_clave([ACTA])[0])
    print()
    for e in reconoce_entidades([ACTA])[0]:
        print(f"  {e['categoria']:14} {e['subcategoria'] or '':10} {e['texto']:35} "
              f"{e['confianza']:.2f}")

## 6. `PiiEntityRecognition`: anonimizar antes de procesar

Esta tarea merece su propio apartado porque es la que hace posible muchos proyectos que de otra
manera no se podrían plantear.

Detecta datos personales en un texto y devuelve una **versión enmascarada**. Con eso puedes
anonimizar antes de enviar el texto al resto del procesamiento, o antes de guardarlo.

No es infalible, y por tanto **no exime de la evaluación de riesgos ni de la base legal**. Pero es
la diferencia entre un proyecto viable y uno que no lo es.

In [ ]:
def anonimiza(textos: Sequence[str], idioma: str = "es",
              categorias: Sequence[str] | None = None) -> list[dict]:
    """Texto enmascarado y lista de datos personales detectados."""
    parametros = {"modelVersion": "latest"}
    if categorias:
        parametros["piiCategories"] = list(categorias)

    respuesta = _analiza("PiiEntityRecognition", textos, idioma, parametros)
    salida = []
    for item in _ordena(respuesta, len(textos)):
        if not item or "error" in item:
            salida.append({"error": (item or {}).get("error", "sin respuesta")})
            continue
        salida.append({
            "texto_enmascarado": item["redactedText"],
            "detectados": [
                {"texto": e["text"], "categoria": e["category"],
                 "confianza": e["confidenceScore"]}
                for e in item.get("entities", [])
            ],
        })
    return salida

In [ ]:
RECLAMACION = (
    "Me llamo Amparo Ferrer Bonet, con DNI 12345678Z, y vivo en la calle Sant Josep 14 "
    "de Alzira. El 3 de febrero llamé al 961234567 y nadie me atendió. Mi correo es "
    "amparo.ferrer@ejemplo.org y mi número de expediente el 2026/AL/00871."
)

if CLAVE and ENDPOINT:
    resultado = anonimiza([RECLAMACION])[0]
    print(resultado["texto_enmascarado"])
    print()
    for d in resultado["detectados"]:
        print(f"  {d['categoria']:22} {d['texto']:32} {d['confianza']:.2f}")

**Léelo con espíritu crítico**, que es lo que se pide en la práctica: mira qué ha detectado y qué
se le ha escapado. Un número de expediente interno no es una categoría estándar y suele pasar sin
detectar, y sin embargo identifica a la persona igual de bien que el DNI dentro de esa
organización.

De ahí la conclusión que hay que sacar: la anonimización automática **reduce** el riesgo, no lo
elimina. Quien decide qué es un dato personal en un contexto concreto sigue siendo una persona.

## 7. Resumen: el patrón asíncrono

El resumen no cabe en una llamada síncrona. El servicio responde **202 Accepted**, pone una URL
en la cabecera `operation-location`, y tú la consultas hasta que el estado sea `succeeded`.

```
tu código  --- POST /analyze-text/jobs -------->  Azure
           <-- 202 Accepted + operation-location --
           --- GET operation-location ---------->
           <-- {"status": "running"} -------------
           --- GET operation-location ---------->
           <-- {"status": "succeeded", ...} ------
```

Dos reglas al implementarlo:

1. **Nunca un `while True`.** Un límite de intentos, y un `TimeoutError` si se agota. Si no, el
   día que un trabajo se quede colgado tu aplicación se cuelga con él.
2. **Espera entre consultas.** Consultar en bucle cerrado no acelera el trabajo y sí consume
   cuota.

In [ ]:
URL_TRABAJOS = f"{ENDPOINT}/language/analyze-text/jobs?api-version=2023-04-01"


def resume(texto: str, frases: int = 3, idioma: str = "es",
           intentos: int = 30, espera: float = 2.0) -> list[str]:
    """Resumen extractivo: devuelve las frases seleccionadas del texto original."""
    cuerpo = {
        "displayName": "Resumen UD2",
        "analysisInput": {"documents": [{"id": "1", "language": idioma, "text": texto}]},
        "tasks": [{
            "kind": "ExtractiveSummarization",
            "taskName": "resumen",
            "parameters": {"sentenceCount": frases},
        }],
    }

    envio = requests.post(URL_TRABAJOS, headers=CABECERAS, json=cuerpo, timeout=20)
    if envio.status_code != 202:
        _comprueba(envio)                      # lanza la excepción que corresponda
        raise ErrorServicio(f"Se esperaba 202 y llegó {envio.status_code}")

    seguimiento = envio.headers["operation-location"]

    for _ in range(intentos):
        time.sleep(espera)
        estado = requests.get(seguimiento, headers=CABECERAS, timeout=20).json()
        if estado["status"] == "succeeded":
            break
        if estado["status"] in ("failed", "cancelled"):
            raise ErrorServicio(f"El trabajo terminó como {estado['status']}: {estado}")
    else:
        raise ErrorProveedor(
            f"El resumen no terminó en {intentos * espera:.0f} segundos"
        )

    tarea = estado["tasks"]["items"][0]
    documento = tarea["results"]["documents"][0]
    return [frase["text"] for frase in documento["sentences"]]

In [ ]:
TEXTO_LARGO = (
    "La inteligencia artificial en la administración pública plantea una tensión que no "
    "es nueva pero sí más aguda. Por un lado, permite tramitar en horas expedientes que "
    "antes tardaban semanas, y liberar personal para las tareas que exigen criterio. "
    "Por otro, introduce decisiones automatizadas cuya lógica no siempre se puede "
    "explicar a la persona afectada, lo que choca con el derecho a una resolución "
    "motivada. La normativa europea ha optado por clasificar los sistemas según su "
    "riesgo, y varios usos administrativos quedan en la categoría de alto riesgo. Eso "
    "obliga a documentar el sistema, a mantener supervisión humana efectiva y a poder "
    "auditar los datos con los que se entrenó. El coste de cumplir esos requisitos es "
    "real, y es la razón por la que muchos proyectos se quedan en piloto."
)

if CLAVE and ENDPOINT:
    for frase in resume(TEXTO_LARGO, frases=3):
        print("-", frase)

Fíjate en qué clase de resumen es: **extractivo**, es decir, selecciona frases del original sin
escribir nada nuevo. Es predecible y no inventa, que en un contexto administrativo es una
ventaja. El resumen **abstractivo** (`AbstractiveSummarization`) redacta de nuevo, se lee mejor y
puede introducir cosas que el original no decía.

Elegir entre los dos no es una cuestión de calidad, es una cuestión de qué riesgo aceptas.

## 8. Contar lo que gastas

En la P2.2 hay que decir cuántas llamadas ha hecho tu cuaderno. No se estima de memoria: se
instrumenta.

In [ ]:
LLAMADAS = {"analyze-text": 0, "jobs": 0, "polling": 0}


def cuenta(clase):
    LLAMADAS[clase] = LLAMADAS.get(clase, 0) + 1


# En tu módulo, la llamada a cuenta() va dentro de _analiza y de resume.
# Aquí solo se enseña el resultado que hay que aportar en la memoria.
print("Llamadas de este cuaderno:", LLAMADAS)
print("Recuerda: lo que se factura es la llamada y el volumen de texto, no el documento.")

## Lo que te llevas a `servicios/lenguaje.py`

Las seis funciones de este cuaderno, tal cual, más `_analiza`, `_comprueba` y `_ordena`. Con dos
cambios que pide la práctica:

1. Las excepciones y la sesión HTTP se mueven a `servicios/http.py`, con los **reintentos** para
   429 y 5xx.
2. `CLAVE` y `ENDPOINT` no son variables de módulo leídas al importar: se leen en una función de
   configuración, para que el módulo se pueda importar sin credenciales y se pueda probar.

Y una comprobación antes de darlo por bueno: **envía cuatro documentos con uno inválido** (una
cadena vacía, o un texto de 200.000 caracteres) y confirma que tu función devuelve tres
resultados y un error identificado, no tres resultados y silencio.

Siguiente: **UD2.04**, imagen.